In [1]:
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()


id,prompt,answer
str,str,str
"""00066667""","""In Alice's Wonderland, a secre…","""10010111"""
"""000b53cf""","""In Alice's Wonderland, a secre…","""01000011"""
"""00189f6a""","""In Alice's Wonderland, secret …","""cat imagines book"""
"""001b24c4""","""In Alice's Wonderland, numbers…","""XXXVIII"""
"""001c63cb""","""In Alice's Wonderland, secret …","""wizard creates secret"""


In [ ]:
import os
import site
import polars as pl
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import kagglehub
import mamba_ssm
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# -----------------------------
# Paths / setup
# -----------------------------
site.addsitedir("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script")
site.addsitedir("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/")

MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
OUTPUT_DIR = "/kaggle/working/lora_adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# Small smoke-test config
# -----------------------------
LORA_RANK = 8
MAX_PROMPT_TOKENS = 192
MAX_ANSWER_TOKENS = 64
MAX_STEPS = 20          # only test a few steps first
GRAD_ACCUM_STEPS = 4
TRAIN_ROWS = 256        # do not start with the full dataset

# -----------------------------
# Data
# -----------------------------
train = pl.read_csv("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv")
train = train.head(TRAIN_ROWS)
records = train.to_dicts()

# -----------------------------
# Tokenizer
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -----------------------------
# 4-bit config
# -----------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# -----------------------------
# Model
# -----------------------------
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# -----------------------------
# LoRA
# -----------------------------
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.train()
model.print_trainable_parameters()

# -----------------------------
# Dataset
# -----------------------------
class PromptAnswerDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]

def collate_fn(batch):
    input_tensors = []
    label_tensors = []

    for row in batch:
        prompt = row["prompt"]
        answer = row["answer"]

        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_PROMPT_TOKENS,
        )["input_ids"]

        answer_ids = tokenizer(
            answer,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_ANSWER_TOKENS,
        )["input_ids"]

        ids = prompt_ids + answer_ids + [tokenizer.eos_token_id]
        labels = [-100] * len(prompt_ids) + answer_ids + [tokenizer.eos_token_id]

        input_tensors.append(torch.tensor(ids, dtype=torch.long))
        label_tensors.append(torch.tensor(labels, dtype=torch.long))

    input_ids = pad_sequence(
        input_tensors,
        batch_first=True,
        padding_value=tokenizer.pad_token_id,
    )
    labels = pad_sequence(
        label_tensors,
        batch_first=True,
        padding_value=-100,
    )
    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

dataset = PromptAnswerDataset(records)
loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)

# -----------------------------
# Optimizer
# -----------------------------
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-5,
)

# -----------------------------
# Train
# -----------------------------
device = next(p for p in model.parameters() if p.device.type != "meta").device
print("Input device:", device)

optimizer.zero_grad(set_to_none=True)

for step, batch in enumerate(loader):
    if step >= MAX_STEPS:
        break

    batch = {k: v.to(device) for k, v in batch.items()}
    outputs = model(**batch)
    loss = outputs.loss / GRAD_ACCUM_STEPS
    loss.backward()

    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    if step % 5 == 0:
        print(f"step={step}, loss={loss.item() * GRAD_ACCUM_STEPS:.4f}")
        if torch.cuda.is_available():
            mem = torch.cuda.memory_allocated() / 1024**3
            print(f"allocated_gb={mem:.2f}")

print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)
`torch_dtype` is deprecated! Use `dtype` instead!
/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/cuda/__init__.py:375: UserWarning: Found GPU0 Tesla P100-PCIE-16GB which is of compute capability (CC) 6.0.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to inst

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')